# 3.8 Exception Handling

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 3: Python for Machine Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-03-python-for-machine-learning/03.08_exception-handling.ipynb)

Companion code for Section 3.8 of the book. Run the cells in order from top to bottom.

**Example 3.8.1**

In [ ]:
# Basic try/except
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print(f'Error: cannot divide {a} by zero')
        return None
    else:
        # else block: runs only if NO exception was raised
        print(f'{a} / {b} = {result}')
        return result
    finally:
        # finally block: ALWAYS runs, exception or not - for cleanup
        print('Division operation complete.')

safe_divide(10, 2)
safe_divide(10, 0)

# Catching multiple exception types
def load_and_process(filepath, column):
    import pandas as pd
    try:
        df = pd.read_csv(filepath)
        values = df[column].dropna().values
        mean_val = values.mean()
        return mean_val
    except FileNotFoundError:
        print(f'Error: file not found: {filepath}')
    except KeyError:
        print(f'Error: column "{column}" not in dataset')
    except pd.errors.EmptyDataError:
        print(f'Error: file {filepath} is empty')
    except Exception as e:
        # Catch-all - always log what happened
        print(f'Unexpected error: {type(e).__name__}: {e}')
    return None

# These calls demonstrate different exceptions
load_and_process('nonexistent.csv', 'age')       # FileNotFoundError

# Raising exceptions
def validate_learning_rate(lr):
    if not isinstance(lr, (int, float)):
        raise TypeError(f'learning_rate must be numeric, got {type(lr).__name__}')
    if lr <= 0 or lr >= 1:
        raise ValueError(f'learning_rate must be in (0,1), got {lr}')
    return lr

try:
    validate_learning_rate(2.5)
except ValueError as e:
    print(f'Caught: {e}')

**Example 3.8.2**

In [ ]:
# Custom exceptions for ML-specific errors
class MLError(Exception):
    '''Base class for all ML-related errors.'''
    pass

class ModelNotFittedError(MLError):
    '''Raised when predict() is called before fit().'''
    def __init__(self, model_name):
        self.model_name = model_name
        super().__init__(f'{model_name} must be fitted before calling predict(). Call fit(X, y) first.')

class DataValidationError(MLError):
    '''Raised when input data fails validation checks.'''
    def __init__(self, message, n_errors=None):
        self.n_errors = n_errors
        super().__init__(message)

class ConvergenceWarning(MLError):
    '''Raised when a model fails to converge.'''
    pass

# Using custom exceptions in a model class
class SafeModel:
    def __init__(self):
        self._is_fitted = False
        self.weights    = None

    def fit(self, X, y):
        import numpy as np
        if X.shape[0] != y.shape[0]:
            raise DataValidationError(
                f'X has {X.shape[0]} samples but y has {y.shape[0]}'
            )
        self.weights    = np.random.randn(X.shape[1])
        self._is_fitted = True
        print('Model fitted successfully.')
        return self

    def predict(self, X):
        if not self._is_fitted:
            raise ModelNotFittedError('SafeModel')
        return X @ self.weights

import numpy as np
model = SafeModel()

# Try predicting before fitting
try:
    preds = model.predict(np.random.randn(10, 5))
except ModelNotFittedError as e:
    print(f'Caught ModelNotFittedError: {e}')

# Try fitting with mismatched data
try:
    model.fit(np.random.randn(100, 5), np.random.randn(90))
except DataValidationError as e:
    print(f'Caught DataValidationError: {e}')

# Correct usage
model.fit(np.random.randn(100, 5), np.random.randn(100))
preds = model.predict(np.random.randn(10, 5))
print(f'Predictions shape: {preds.shape}')